# Clase 09 — Archivos: Leer y Escribir Datos

## Objetivos de la clase

Al terminar esta clase vas a poder:

- Abrir archivos con `open()` entendiendo los modos `'r'`, `'w'`, `'a'` y `'x'`.
- Usar `with` como la forma segura y estándar de trabajar con archivos.
- Leer contenido con `read()`, `readline()` y `readlines()`, y saber cuándo usar cada uno.
- Escribir y anexar contenido con `write()`.
- Copiar o transformar el contenido de un archivo hacia otro.
- Controlar el cursor de un archivo con `seek()` y `tell()`, y usar los modos combinados `r+`, `w+`, `a+`.

## ¿Por qué nos importa esto?

Hasta ahora los datos vivían dentro de tu código: los escribiste tú en una lista o variable. En la vida real, los datos llegan en archivos — un `.csv` con ventas, un `.txt` con logs, un `.json` de una API guardada en disco. Antes de limpiar, analizar o graficar nada, hay que saber sacar esa información de un archivo y meterla en Python (y, al revés, guardar resultados). `pandas` y `numpy` hacen exactamente esto por debajo, con comodidades encima — entender lo de abajo evita muchos dolores de cabeza más adelante.

## 1. `open()` y sus modos: la puerta de entrada a cualquier archivo

`open()` recibe dos cosas: la ruta del archivo y el **modo**. Si no indicas modo, Python asume `'r'`.

| Modo | Qué hace |
|---|---|
| `'r'` | Lee un archivo existente. Si no existe, lanza un error. |
| `'w'` | Escribe. Crea el archivo si no existe, y **borra todo el contenido** si ya existía. |
| `'a'` | Anexa (append). Escribe al final sin borrar nada. Crea el archivo si no existe. |
| `'x'` | Creación exclusiva. Crea el archivo, pero lanza error si ya existe. |

El error más común de quien empieza: confundir `'w'` con `'a'`. Abrir en `'w'` para "agregar algo" en realidad vacía el archivo primero.

In [ ]:
# 'w' sobrescribe: si 'notas.txt' ya tenía contenido, desaparece
with open('notas.txt', 'w') as archivo:
    archivo.write("Esto es lo único que va a quedar\n")

with open('notas.txt', 'r') as archivo:
    print(archivo.read())

In [ ]:
# 'a' respeta lo que ya había y agrega después
with open('notas.txt', 'a') as archivo:
    archivo.write("Esto se agrega al final, sin borrar nada\n")

with open('notas.txt', 'r') as archivo:
    print(archivo.read())

## 2. `with`: por qué no deberías abrir un archivo "a mano"

Cuando abres un archivo con `open()`, Python te da un **objeto archivo** — como una llave prestada por el sistema operativo. Si no la devuelves (`close()`), esos recursos quedan atascados, y si el programa se cae por una excepción a mitad de camino, el archivo nunca se cierra correctamente.

In [ ]:
# Forma manual: funciona, pero es riesgosa
archivo = open('notas.txt', 'r')
contenido = archivo.read()
archivo.close()  # si algo falla ANTES de esta línea, el archivo nunca se cierra
print(contenido)

In [ ]:
# Forma correcta y recomendada: with actúa como context manager
with open('notas.txt', 'r') as archivo:
    contenido = archivo.read()
# Al salir del bloque, el archivo se cierra solo, pase lo que pase adentro

print(archivo.closed)  # True -> with ya lo cerró por ti
print(archivo.name)    # 'notas.txt'
print(archivo.mode)    # 'r'

## 3. Tres formas de leer: `read()`, `readline()` y `readlines()`

**`read()` — trae todo de una vez.** Devuelve todo el contenido como un único string, con los `\n` incluidos.

In [ ]:
# Creamos un archivo con varias líneas para los ejemplos que siguen
with open('datos.txt', 'w') as archivo:
    archivo.write("primera linea\n")
    archivo.write("segunda linea\n")
    archivo.write("tercera linea\n")

with open('datos.txt', 'r') as archivo:
    contenido = archivo.read()
    print(contenido)
    print(type(contenido))

In [ ]:
# read(n) lee los siguientes n caracteres desde donde está el cursor.
# Es acumulativo: cada llamada continúa donde quedó la anterior.
with open('datos.txt', 'r') as archivo:
    print(archivo.read(4))   # los primeros 4 caracteres
    print(archivo.read(4))   # los siguientes 4 caracteres
    print(archivo.read(7))   # los siguientes 7 caracteres

**`readline()` — una línea a la vez.** Cada llamada devuelve la siguiente línea como string.

In [ ]:
with open('datos.txt', 'r') as archivo:
    primera = archivo.readline()
    segunda = archivo.readline()
    print(primera)
    print(segunda)

In [ ]:
# Para leer todo el archivo sin saber cuántas líneas tiene:
# el bucle corta cuando readline() devuelve una cadena vacía (que Python interpreta como False)
with open('datos.txt', 'r') as archivo:
    while True:
        linea = archivo.readline()
        if not linea:
            break  # ya no quedan líneas
        print(linea)

In [ ]:
# Forma más directa y la más usada en la práctica: iterar sobre el propio objeto archivo
with open('datos.txt', 'r') as archivo:
    for i, linea in enumerate(archivo):
        print("Línea", i, ":", linea)

**`readlines()` — todas las líneas, pero como lista.** Útil cuando necesitas acceso indexado (la línea 5, la penúltima, etc.).

In [ ]:
with open('datos.txt', 'r') as archivo:
    lista_lineas = archivo.readlines()

print(lista_lineas)
print(lista_lineas[0])  # la primera línea
print(lista_lineas[1])  # la segunda línea

**Resumen para decidir:** archivo chico y quieres todo el texto de una → `read()`. Necesitas recorrer línea por línea sin cargar todo en memoria → `for linea in archivo` o `readline()` en un bucle. Necesitas las líneas como lista indexable → `readlines()`.

## 4. Escribir y anexar: `write()`, modo `'w'` y modo `'a'`

A diferencia de `print()`, `write()` **no agrega automáticamente un salto de línea** — si no pones tú el `\n`, todo el texto queda pegado.

In [ ]:
# Crear (o sobrescribir) un archivo y escribir dos líneas
with open('reporte.txt', 'w') as archivo:
    archivo.write("Esta es la línea A\n")
    archivo.write("Esta es la línea B\n")

with open('reporte.txt', 'r') as archivo:
    print(archivo.read())

In [ ]:
# Cuando tienes varias líneas en una lista, lo natural es recorrerla con un for
lineas = ["Primera fila", "Segunda fila", "Tercera fila"]

with open('reporte.txt', 'w') as archivo:
    for linea in lineas:
        archivo.write(linea + "\n")

with open('reporte.txt', 'r') as archivo:
    print(archivo.read())

Cada vez que ejecutas ese bloque con modo `'w'`, el archivo se crea desde cero. Si en cambio quieres **conservar** lo que ya está y solo sumar información nueva (por ejemplo, un log), usas el modo `'a'`.

In [ ]:
nuevo_dato = "Esta es la línea C"

with open('reporte.txt', 'a') as archivo:
    archivo.write(nuevo_dato + "\n")

with open('reporte.txt', 'r') as archivo:
    print(archivo.read())

Un patrón muy habitual en ciencia de datos es copiar o transformar el contenido de un archivo hacia otro: lees línea por línea del origen y escribes en el destino, anidando dos `with`. Este patrón es la base de cualquier proceso que filtra, limpia o reorganiza datos.

In [ ]:
# Preparamos un archivo de origen
with open('origen.txt', 'w') as archivo:
    archivo.write("dato 1\n")
    archivo.write("dato 2\n")
    archivo.write("dato 3\n")

# Copiamos su contenido a un archivo destino, línea por línea
with open('origen.txt', 'r') as archivo_origen:
    with open('destino.txt', 'w') as archivo_destino:
        for linea in archivo_origen:
            archivo_destino.write(linea)

with open('destino.txt', 'r') as archivo:
    print(archivo.read())

## 5. `seek()`, `tell()` y los modos combinados (`r+`, `w+`, `a+`)

Todo archivo abierto tiene un **cursor** invisible que marca desde dónde se lee o escribe a continuación.

- `archivo.tell()` — posición actual del cursor (en bytes).
- `archivo.seek(offset, desde)` — mueve el cursor. `desde`: `0` inicio, `1` posición actual, `2` final.

In [ ]:
with open('reporte.txt', 'r') as archivo:
    print(archivo.tell())        # 0, recién abierto
    archivo.read(10)
    print(archivo.tell())        # 10, avanzó 10 bytes
    archivo.seek(0, 0)           # volver al principio
    print(archivo.tell())        # 0 otra vez

Los modos combinados dejan leer y escribir sobre el mismo archivo abierto:

- **`r+`** — lectura y escritura sobre un archivo existente. No lo vacía, pero tampoco lo crea si no existe.
- **`w+`** — escritura y lectura, pero **trunca** el archivo al abrirlo (igual que `'w'` solo).
- **`a+`** — escritura (siempre al final) y lectura. Crea el archivo si no existe.

La trampa con `a+`: aunque puedas leer, el cursor de escritura siempre queda al final, así que si llamas a `read()` sin mover el cursor primero, es común no obtener nada.

In [ ]:
with open('reporte.txt', 'a+') as archivo:
    archivo.write("Una línea más\n")
    contenido = archivo.read()   # probablemente vacío: el cursor quedó al final
    if not contenido:
        print("No leyó nada porque el cursor está al final")

    archivo.seek(0, 0)           # ahora sí, volvemos al principio
    print(archivo.read())        # esto sí muestra todo el contenido

Con `r+` puedes reescribir contenido sin borrar el resto del archivo — pero si el texto nuevo es más corto que el que reemplaza, sobran caracteres viejos al final. Para eso existe `truncate()`: corta el archivo justo en la posición donde está el cursor.

In [ ]:
with open('reporte.txt', 'r+') as archivo:
    archivo.seek(0, 0)
    archivo.write("Línea 1\n")
    archivo.write("Línea 2\n")
    archivo.truncate()   # borra cualquier resto de contenido viejo después de este punto
    archivo.seek(0, 0)
    print(archivo.read())

## Resumen: lo que te llevas de este capítulo

- `open(ruta, modo)` abre un archivo. Modos básicos: `'r'` (leer), `'w'` (escribir, sobrescribe todo) y `'a'` (anexar, conserva lo existente).
- Usa siempre `with open(...) as archivo:` en vez de `open()` + `close()` manual. Garantiza el cierre incluso si hay una excepción.
- `read()` trae todo como un string (o los siguientes `n` caracteres). Úsalo con archivos chicos.
- `readline()` lee una línea a la vez; combinado con un bucle, recorre el archivo entero sin cargarlo todo en memoria.
- `readlines()` devuelve todas las líneas como lista, útil para acceso indexado.
- `write()` **no** agrega saltos de línea automáticamente — hay que incluir `\n`.
- Copiar/filtrar entre archivos: abre el origen en `'r'` y el destino en `'w'`/`'a'`, y recorre línea por línea.
- `tell()` dice dónde está el cursor; `seek(offset, desde)` lo mueve. Los modos combinados (`r+`, `w+`, `a+`) permiten leer y escribir el mismo archivo abierto, y `truncate()` corta lo que sobra después de la posición actual.